# Generate processed datasets from 34_features_newvol
This notebook converts `34_features_newvol/train_features_5m_clean.parquet` into three datasets:
1. Remove the single monthly peak in `target_ann_vol` (e.g., the April tariff spike).
2. Winsorize numeric columns at the 1% and 99% percentiles.
3. Split into high/low volatility regimes by median `target_ann_vol`.
Outputs are written to `34_features_newvol/processed_sets`.


In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / "Data").exists():
    PROJECT_ROOT = PROJECT_ROOT
elif (PROJECT_ROOT / "Notebooks").exists():

    PROJECT_ROOT = PROJECT_ROOT.parent

else:print('OUTDIR =', OUTDIR)

    PROJECT_ROOT = PROJECT_ROOTprint('INPUT =', INPUT)



DATA_DIR = PROJECT_ROOT / "Data"OUTDIR = DATA_DIR / "processed_sets"
INPUT = DATA_DIR / "train_features_5m_clean.parquet"

INPUT = ..\34_features_newvol\train_features_5m_clean.parquet
OUTDIR = ..\34_features_newvol\processed_sets


In [2]:
def load_df(path=INPUT):
    df = pd.read_parquet(path)
    if 'bar5' in df.columns:
        df['bar5'] = pd.to_datetime(df['bar5'])
    return df

# quick load test
df = load_df()
df.shape


(14004, 34)

In [3]:
def remove_april_peak(df, date_col='bar5'):
    start = pd.Timestamp('2025-04-02')
    end = pd.Timestamp('2025-04-10')
    mask = (df[date_col] < start) | (df[date_col] > end)
    df_removed = df[mask].copy()
    return df_removed, start.date(), end.date()

ds1, removed_start, removed_end = remove_april_peak(df)
removed_start, removed_end, ds1.shape


(datetime.date(2025, 4, 2), datetime.date(2025, 4, 10), (13788, 34))

In [4]:
def winsorize_df(df, col='target_ann_vol', lower_q=0.01, upper_q=0.99):
    lower = float(df[col].quantile(lower_q))
    upper = float(df[col].quantile(upper_q))
    df_w = df.copy()
    df_w[col] = df_w[col].clip(lower=lower, upper=upper)
    return df_w, {col: lower}, {col: upper}

df_w, lq, uq = winsorize_df(df)
df_w.shape

(14004, 34)

In [5]:
def split_regimes(df, target_col='target_ann_vol', date_col='bar5'):
    daily_target = df.groupby(df[date_col].dt.floor('D'))[target_col].mean()
    thr = float(daily_target.median())
    high_days = daily_target[daily_target > thr].index
    is_high_day = df[date_col].dt.floor('D').isin(high_days)
    high = df[is_high_day].copy()
    low = df[~is_high_day].copy()
    return high, low, thr

high, low, thr = split_regimes(df)
thr, high.shape, low.shape

(0.0710187544644155, (7148, 34), (6856, 34))

In [6]:
def summarize_and_save(df, name):
    os.makedirs(OUTDIR, exist_ok=True)
    p = os.path.join(OUTDIR, f'{name}.parquet')
    df.to_parquet(p)
    return p

# Save the three datasets
p1 = summarize_and_save(ds1, 'removed_peak_month')
p2 = summarize_and_save(df_w, 'winsorized_1_99')
p_high = summarize_and_save(high, 'regime_high_vol')
p_low = summarize_and_save(low, 'regime_low_vol')
p1, p2, p_high, p_low


('..\\34_features_newvol\\processed_sets\\removed_peak_month.parquet',
 '..\\34_features_newvol\\processed_sets\\winsorized_1_99.parquet',
 '..\\34_features_newvol\\processed_sets\\regime_high_vol.parquet',
 '..\\34_features_newvol\\processed_sets\\regime_low_vol.parquet')

In [12]:
summary = {
    'original_rows': int(len(df)),
    'original_cols': int(df.shape[1]),
    'removed_peak_month': {'rows': int(len(ds1)), 'path': p1},
    'winsorized_1_99': {'rows': int(len(df_w)), 'path': p2},
    'regime_threshold_target_ann_vol': thr,
    'regime_high_vol': {'rows': int(len(high)), 'path': p_high},
    'regime_low_vol': {'rows': int(len(low)), 'path': p_low},
}
with open(os.path.join(OUTDIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

summary


{'original_rows': 14004,
 'original_cols': 34,
 'removed_peak_month': {'rows': 13788,
  'path': '..\\34_features_newvol\\processed_sets\\removed_peak_month.parquet'},
 'winsorized_1_99': {'rows': 14004,
  'path': '..\\34_features_newvol\\processed_sets\\winsorized_1_99.parquet'},
 'regime_threshold_target_ann_vol': 0.06823979044724975,
 'regime_high_vol': {'rows': 7002,
  'path': '..\\34_features_newvol\\processed_sets\\regime_high_vol.parquet'},
 'regime_low_vol': {'rows': 7002,
  'path': '..\\34_features_newvol\\processed_sets\\regime_low_vol.parquet'}}

In [ ]:
import pandas as pd
from pathlib import Path

# load data if not already loaded
INPUT = DATA_DIR / "train_features_5m_clean.parquet"
df = pd.read_parquet(INPUT)

# removed-peak period from the notebook
start = pd.Timestamp('2025-04-02')
end = pd.Timestamp('2025-04-10')
mask_removed_peak = (df['bar5'] >= start) & (df['bar5'] <= end)

# winsorize extremes outside 1%-99%
q1 = df['target_ann_vol'].quantile(0.01)
q99 = df['target_ann_vol'].quantile(0.99)
mask_winsorized = (df['target_ann_vol'] < q1) | (df['target_ann_vol'] > q99)

# counts
removed_peak_rows = df[mask_removed_peak]
winsorized_rows = df[mask_winsorized]
overlap_rows = df[mask_removed_peak & mask_winsorized]

print('total rows:', len(df))
print('removed-peak rows:', len(removed_peak_rows))
print('winsorized extreme rows:', len(winsorized_rows))
print('overlap rows:', len(overlap_rows))
print('fraction of removed-peak rows also winsorized:', len(overlap_rows) / len(removed_peak_rows))
print('fraction of winsorized rows in removed-peak:', len(overlap_rows) / len(winsorized_rows))

total rows: 14004
removed-peak rows: 216
winsorized extreme rows: 282
overlap rows: 88
fraction of removed-peak rows also winsorized: 0.4074074074074074
fraction of winsorized rows in removed-peak: 0.3120567375886525
